In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 23, Cartan structure equations

Companion notebook to [23_cartan_structure_equations.md](23_cartan_structure_equations.md). `CartanStructureProblem(∇, F)` proves both Cartan I and II mechanically:

$$T^a = de^a + \sum_b \omega^a{}_b \wedge e^b \quad\text{(I)}$$
$$R^a{}_b = d\omega^a{}_b + \sum_c \omega^a{}_c \wedge \omega^c{}_b \quad\text{(II)}$$

## The three form atoms

`ConnectionForm` (`ω^a_b`), `TorsionForm` (`T^a`), `CurvatureForm` (`R^a_b`), all inert until their definitions fire. All carry degree 1.

In [ ]:
from jacopy.calculus.connection import AffineConnection
from jacopy.calculus.local_frame import LocalFrame
from jacopy.calculus.cartan_forms import (
    ConnectionForm, TorsionForm, CurvatureForm,
)

nabla = AffineConnection("∇")
F     = LocalFrame("F", dim=3)

omega_ab = ConnectionForm(nabla, F, "a", "b")
T_a      = TorsionForm   (nabla, F, "a")
R_ab     = CurvatureForm (nabla, F, "a", "b")

print(f"connection form : {omega_ab}")
print(f"torsion form    : {T_a}")
print(f"curvature form  : {R_ab}")

## `CartanStructureProblem`

24-rule engine bundling torsion / curvature unfolding, connection axioms, frame decomposition, indexed-sum machinery, wedge expansion, intrinsic `d`, frame duality. Per-problem registry declares `FrameCovector` and `ConnectionForm` as degree 1.

In [ ]:
from jacopy.library.cartan_structure import CartanStructureProblem

prob = CartanStructureProblem(nabla, F)
print(f"name           : {prob.name}")
print(f"engine rules   : {len(prob.engine.definitions)}")

## LHS / RHS builders

`first_cartan_lhs` / `_rhs` (and their second-equation counterparts) build the side terms as `MultiEval`s on a `(U, V)` pair. The bound dummies (`b` in I, `c` in II) are minted fresh per call.

In [ ]:
from jacopy.algebra.derivation import Derivation

U, V = Derivation("U", 0), Derivation("V", 0)

lhs1 = prob.first_cartan_lhs(U, V, "a")
rhs1 = prob.first_cartan_rhs(U, V, "a")
print(f"Cartan I LHS : {lhs1}")
print(f"Cartan I RHS : {rhs1}")

## `prove_first_cartan`, Cartan I

~49 steps cover torsion-form opening, frame decomposition of `U`/`V`, Y-Leibniz on each `∇_U V`, connection-form decomposition of `∇_V X_b`, intrinsic `d` on `e^a`, wedge alternating expansion, and Kronecker contractions + pairing duality collapsing `⟨e^a, X_b⟩` matches.

In [ ]:
res1 = prob.prove_first_cartan(U, V, "a")
print(f"Cartan I  : ok={res1.ok}, steps={len(res1.steps)}")

## `prove_second_cartan`, Cartan II

~54 steps. Structurally similar to Cartan I, both are shadows of the same abstract identity.

In [ ]:
res2 = prob.prove_second_cartan(U, V, "a", "b")
print(f"Cartan II : ok={res2.ok}, steps={len(res2.steps)}")

## When to use what

| If you want… | Reach for… |
|---|---|
| Index-laden Cartan structure equation | `CartanStructureProblem` |
| Coordinate-free Bianchi (no frame) | `BianchiProblem` (tutorial 20) |
| Generic operator equality on forms | `prove_intrinsic_equivalence` (tutorial 15) |
| Q9 Koszul mode (custom bracket on `∇`) | Same wrapper, auto-detects via `_intrinsic_d_rule` |

## Summary

* `ConnectionForm`, `TorsionForm`, `CurvatureForm`, three inert degree-1 form atoms; definitions in `cartan_forms` engine rules.
* `CartanStructureProblem(∇, F)`, 24-rule engine spanning seven phases (torsion/curvature, connection axioms, frame decomp, indexed-sum, wedge, intrinsic d, duality).
* `prove_first_cartan` ~49 steps; `prove_second_cartan` ~54 steps. Both run engine + simplify (with `sort_product`) to a fix-point.
* Auto-detects custom-bracket connections (Q9) and swaps in `KoszulExteriorDIntrinsicDefinition`, no caller action needed.
* Skip when the problem is coordinate-free (no frame, no index).